# Fix baseline TensorBoard loss overlays

This standalone notebook mounts Google Drive, rebuilds overlaid train/validation loss summaries from the saved baseline checkpoint, and opens TensorBoard. It does not download or open the training dataset.


In [ ]:
from pathlib import Path
import re

import torch
from torch.utils.tensorboard import SummaryWriter
from google.colab import drive

drive.mount('/content/drive')

BASELINE_DIR = Path('/content/drive/MyDrive/baseline')
MODEL_DIR = BASELINE_DIR / 'models'
LOG_DIR = BASELINE_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def select_checkpoint(model_dir: Path) -> Path:
    latest = model_dir / 'baseline_latest.pth'
    if latest.exists():
        return latest
    best = model_dir / 'best_baseline.pth'
    if best.exists():
        return best
    epoch_checkpoints = []
    for path in model_dir.glob('epoch_*_baseline.pth'):
        match = re.fullmatch(r'epoch_(\d+)_baseline\.pth', path.name)
        if match:
            epoch_checkpoints.append((int(match.group(1)), path))
    if not epoch_checkpoints:
        raise FileNotFoundError(f'No baseline checkpoint found in {model_dir}')
    return max(epoch_checkpoints, key=lambda item: item[0])[1]

CHECKPOINT = select_checkpoint(MODEL_DIR)
print(f'checkpoint: {CHECKPOINT}')
print(f'logs: {LOG_DIR}')


In [ ]:
try:
    checkpoint = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT, map_location='cpu')

history = checkpoint.get('history', {})
required = (
    'train_total', 'val_total',
    'train_motion', 'val_motion',
    'train_gripper', 'val_gripper',
)
missing = [key for key in required if key not in history]
if missing:
    raise ValueError(f'Checkpoint history is missing: {missing}')

num_epochs = min(len(history[key]) for key in required)
if num_epochs == 0:
    raise ValueError('Checkpoint history contains no completed epochs')

writer = SummaryWriter(log_dir=LOG_DIR)
for epoch in range(num_epochs):
    writer.add_scalars(
        'loss/total',
        {'train': history['train_total'][epoch],
         'val': history['val_total'][epoch]},
        epoch,
    )
    writer.add_scalars(
        'loss/motion',
        {'train': history['train_motion'][epoch],
         'val': history['val_motion'][epoch]},
        epoch,
    )
    writer.add_scalars(
        'loss/gripper',
        {'train': history['train_gripper'][epoch],
         'val': history['val_gripper'][epoch]},
        epoch,
    )
writer.close()

print(f'created combined total, motion, and gripper panels for {num_epochs} epochs')
print('No dataset was accessed.')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/baseline/logs
